In [9]:
# general
import pandas as pd
from sklearn.metrics import accuracy_score

In [10]:
k2_key = "sk-your-openai-key-here"

In [11]:
from openai import OpenAI

client = OpenAI(api_key=k2_key)
prompt = input("Enter Prompt: ")
# 2. send a message and get a response
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt}
    ]
)

# 3. extract the text
print("\n",response.choices[0].message.content)
# Output: Dar es Salaam

Enter Prompt:  blaa



 It seems like you might have entered something by mistake. How can I assist you today?


In [12]:
df = pd.read_csv("tz_eac_economic_headlines.csv")
df_comparison = df[["headline","sentiment"]].copy() 
df.head()

,date,headline,source,category,sentiment,tzs_usd_rate,inflation_rate,bot_policy_rate
0,2021-01-05,Bank of Tanzania holds policy rate steady amid...,Daily News,Policy,Neutral,2296,3.1,5.0
1,2021-01-08,Tanzanian shilling opens week firmer as dollar...,The Citizen,Forex,Positive,2291,3.1,5.0
2,2021-01-12,KCB Tanzania reports 18% rise in mobile loan d...,Business Daily Africa,Fintech,Positive,2294,3.1,5.0
3,2021-01-15,Tanzania Revenue Authority misses January targ...,The EastAfrican,Taxation,Negative,2297,3.2,5.0
4,2021-01-19,Cashew export volumes from Mtwara port fall sh...,Mwananchi,Agriculture,Negative,2300,3.2,5.0


In [ ]:
#pip install groq

In [14]:
from groq import Groq
import time

groq_client = Groq(api_key="gsk-your-groq-key-here")


system_prompt = """You are classifying the ECONOMIC IMPACT of East African financial news headlines.

Positive:
Likely improves growth, stability, investment, consumer conditions, or business activity.

Negative:
announcements, budget approvals, and new guidelines where no outcome is speculated yet, or steady state with no directional signal

Neutral:
Informational updates, announcements, or events without a clear positive or negative economic implication.

Focus on the likely economic effect implied by the headline, not emotional tone.

Do not return the statement you just analaysed just output a final one word nothing more.

Return only:
1. Positive
2. Negative
3. Neutral
""" 
models = {
    "gpt-4o-mini": "openai",
    "llama-3.1-8b-instant": "groq",
    "llama-3.3-70b-versatile": "groq",
}

def multi_model_sentiment_batch(headlines_df, models=models, batch_size=25, system_prompt=system_prompt):
    headlines = headlines_df.tolist()
    VALID_LABELS = {"Positive", "Negative", "Neutral"}
    max_retries = 3
    results_dict = {}

    for model_name, provider in models.items():
        print(f"\n{'='*50}")
        print(f"Running model: {model_name} ({provider})")
        print(f"{'='*50}")

        # pick the right client
        api_client = openai_client if provider == "openai" else groq_client

        for attempt in range(1, max_retries + 1):
            print(f"\nAttempt {attempt}...\n")
            all_results = []
            batches = [
                headlines[i:i+batch_size]
                for i in range(0, len(headlines), batch_size)
            ]

            for batch_num, batch in enumerate(batches):
                print(f"Processing batch {batch_num + 1}/{len(batches)} ({len(batch)} headlines)...")
                numbered = "\n".join(
                    [f"{i+1}. {h}" for i, h in enumerate(batch)]
                )

                try:
                    response = api_client.chat.completions.create(
                        model=model_name,
                        messages=[
                            {"role": "system", "content": system_prompt},
                            {"role": "user", "content": numbered}
                        ],
                        temperature=0
                    )
                    lines = response.choices[0].message.content.strip().split("\n")
                    batch_results = [
                        line.split(". ", 1)[1].strip()
                        for line in lines
                        if ". " in line
                    ]
                    all_results.extend(batch_results)

                except Exception as e:
                    print(f"Batch {batch_num + 1} failed: {e}")
                    all_results.extend(["Error"] * len(batch))

                if batch_num < len(batches) - 1:
                    time.sleep(1)

            correct_length = len(all_results) == len(headlines)
            valid_labels = set(all_results).issubset(VALID_LABELS)
            print(f"\nOutput count: {len(all_results)} / {len(headlines)}")
            print(f"Unique outputs: {set(all_results)}")

            if correct_length and valid_labels:
                print(f"Validation passed for {model_name}.")
                results_dict[model_name] = all_results
                break

            print("Validation failed. Retrying...\n")

        else:
            print(f"{model_name} failed after {max_retries} attempts. Storing None.")
            results_dict[model_name] = None

    # build comparison dataframe
    comparison_df = headlines_df.reset_index(drop=True).to_frame(name="headline")
    for model_name, preds in results_dict.items():
        comparison_df[model_name] = preds if preds else "Failed"

    return comparison_df

In [8]:
# df_comparison["groq_results"] = groq_sentiment_batch(df_comparison["headline"], 50)
# df_comparison.head()

In [9]:
# true_labels = df_comparison['sentiment']
# groq_labels = df_comparison['groq_results']

# print(f"Open ai accuracy:   {accuracy_score(true_labels, groq_labels):.2%}")

In [ ]:
"""You are classifying the ECONOMIC IMPACT of East African financial news headlines.

Positive:
Likely improves growth, stability, investment, consumer conditions, or business activity.

Negative:
Indicates contraction, financial loss, increased risk, market decline, or unfavorable regulatory/legal changes.

Neutral:
Informational updates, budget approvals, and new guidelines where no outcome is speculated yet, or steady state with no directional signal.

Focus on the likely economic effect implied by the headline, not emotional tone.

Do not return the statement you just analaysed just output a final one word nothing more.

Return only:
1. Positive
2. Negative
3. Neutral
""" 

In [24]:
from openai import OpenAI
from groq import Groq

k2_key = "sk-your-openai-key-here"
openai_client = OpenAI(api_key=k2_key)
groq_client = Groq(api_key="gsk-your-groq-key-here")


models = {
    "gpt-4o-mini": "openai",
    "llama-3.1-8b-instant": "groq",
    "llama-3.3-70b-versatile": "groq",
}

system_prompt = """You are classifying the ECONOMIC IMPACT of East African financial news headlines.

Positive:
Likely improves growth, stability, investment, consumer conditions, or business activity.

Negative:
Indicates contraction, financial loss, increased risk, market decline, or unfavorable regulatory/legal changes.

Neutral:
Informational updates, budget approvals, and new guidelines where no outcome is speculated yet, or steady state with no directional signal.

Focus on the likely economic effect implied by the headline, not emotional tone.

Do not return the statement you just analaysed just output a final one word nothing more.

Return only:
1. Positive
2. Negative
3. Neutral
""" 



def multi_model_sentiment_batch(headlines_df,  batch_size=25, models=models,system_prompt=system_prompt):
    headlines = headlines_df.tolist()
    VALID_LABELS = {"Positive", "Negative", "Neutral"}
    max_retries = 3
    results_dict = {}

    for model_name, provider in models.items():
        print(f"\n{'='*50}")
        print(f"Running model: {model_name} ({provider})")
        print(f"{'='*50}")

        # pick the right client
        api_client = openai_client if provider == "openai" else groq_client

        for attempt in range(1, max_retries + 1):
            print(f"\nAttempt {attempt}...\n")
            all_results = []
            batches = [
                headlines[i:i+batch_size]
                for i in range(0, len(headlines), batch_size)
            ]

            for batch_num, batch in enumerate(batches):
                print(f"Processing batch {batch_num + 1}/{len(batches)} ({len(batch)} headlines)...")
                numbered = "\n".join(
                    [f"{i+1}. {h}" for i, h in enumerate(batch)]
                )

                try:
                    response = api_client.chat.completions.create(
                        model=model_name,
                        messages=[
                            {"role": "system", "content": system_prompt},
                            {"role": "user", "content": numbered}
                        ],
                        temperature=0
                    )
                    lines = response.choices[0].message.content.strip().split("\n")
                    batch_results = [
                        line.split(". ", 1)[1].strip()
                        for line in lines
                        if ". " in line
                    ]
                    all_results.extend(batch_results)

                except Exception as e:
                    print(f"Batch {batch_num + 1} failed: {e}")
                    all_results.extend(["Error"] * len(batch))

                if batch_num < len(batches) - 1:
                    time.sleep(1)

            correct_length = len(all_results) == len(headlines)
            valid_labels = set(all_results).issubset(VALID_LABELS)
            print(f"\nOutput count: {len(all_results)} / {len(headlines)}")
            # print(f"Unique outputs: {set(all_results)}")

            if correct_length and valid_labels:
                print(f"Validation passed for {model_name}.")
                results_dict[model_name] = all_results
                break

            print("Validation failed. Retrying...\n")

        else:
            print(f"{model_name} failed after {max_retries} attempts. Storing None.")
            results_dict[model_name] = None

    # build comparison dataframe
    comparison_df = headlines_df.reset_index(drop=True).to_frame(name="headline")
    for model_name, preds in results_dict.items():
        comparison_df[model_name] = preds if preds else "Failed"

    return comparison_df

In [25]:
comparison_df = multi_model_sentiment_batch(df["headline"], 100)
comparison_df.head()


Running model: gpt-4o-mini (openai)

Attempt 1...

Processing batch 1/6 (100 headlines)...
Processing batch 2/6 (100 headlines)...
Processing batch 3/6 (100 headlines)...
Processing batch 4/6 (100 headlines)...
Processing batch 5/6 (100 headlines)...
Processing batch 6/6 (4 headlines)...

Output count: 304 / 504
Validation failed. Retrying...


Attempt 2...

Processing batch 1/6 (100 headlines)...
Processing batch 2/6 (100 headlines)...
Processing batch 3/6 (100 headlines)...
Processing batch 4/6 (100 headlines)...
Processing batch 5/6 (100 headlines)...
Processing batch 6/6 (4 headlines)...

Output count: 404 / 504
Validation failed. Retrying...


Attempt 3...

Processing batch 1/6 (100 headlines)...
Processing batch 2/6 (100 headlines)...
Processing batch 3/6 (100 headlines)...
Processing batch 4/6 (100 headlines)...
Processing batch 5/6 (100 headlines)...
Processing batch 6/6 (4 headlines)...

Output count: 404 / 504
Validation failed. Retrying...

gpt-4o-mini failed after 3 attemp

,headline,gpt-4o-mini,llama-3.1-8b-instant,llama-3.3-70b-versatile
0,Bank of Tanzania holds policy rate steady amid...,Failed,Failed,Failed
1,Tanzanian shilling opens week firmer as dollar...,Failed,Failed,Failed
2,KCB Tanzania reports 18% rise in mobile loan d...,Failed,Failed,Failed
3,Tanzania Revenue Authority misses January targ...,Failed,Failed,Failed
4,Cashew export volumes from Mtwara port fall sh...,Failed,Failed,Failed


In [21]:
true_labels = df_comparison['sentiment']

gpt_4o_mini_sentiments = comparison_df['gpt-4o-mini']
llama_instant_sentiments = comparison_df['llama-3.1-8b-instant']
llama_versatile_sentiments = comparison_df['llama-3.3-70b-versatile']

print(f"GPT 40 mini accuracy:   {accuracy_score(true_labels, gpt_4o_mini_sentiments):.2%}")
print(f"llama 3.1 8b instant accuracy:   {accuracy_score(true_labels, llama_instant_sentiments):.2%}")
print(f"llama 3.370b versatile accuracy:   {accuracy_score(true_labels, llama_versatile_sentiments):.2%}")

GPT 40 mini accuracy:   93.65%
llama 3.1 8b instant accuracy:   90.08%
llama 3.370b versatile accuracy:   91.27%
